In [ ]:
import re
from pathlib import Path
import pandas as pd
from rapidfuzz import process, fuzz
from unidecode import unidecode
from tqdm import tqdm
import yaml

In [ ]:
import os
os.chdir('../../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## Import WOS affiliation name

In [ ]:
wos = pd.read_csv(dataset_config['path_processed'] + 'WOS/WOS_CNfirm_publication.csv')
wos

In [ ]:
wos_aff_stdname = pd.read_csv(dataset_config['path_processed'] + 'WOS/WOS_CNfirm_publication.csv', usecols=['affiliationame']).drop_duplicates()
wos_aff_stdname

## Import CSMAR and subs eng name from Qichacha

In [ ]:
Qichacha_sbus_eng = pd.read_csv(dataset_config['path_processed'] + 'Qichacha_firm_subs_eng.csv') #, nrows=1000
Qichacha_sbus_eng

## Fuzzy Matching

In [ ]:
# ===============================
# 0. Imports
# ===============================
# !pip install rapidfuzz tqdm

import re
import pandas as pd
from rapidfuzz import process, fuzz
from tqdm import tqdm

tqdm.pandas()  # enable progress bar for pandas apply


# ======================================
# 1. Cleaning function for EN company names
# ======================================
def clean_company_name_en(name: str) -> str:
    """
    Clean English company names:
    - lowercase
    - normalize punctuation to spaces
    - remove common legal suffixes (Ltd, Inc, Co, Corp, Group, etc.)
    - collapse multiple spaces
    """
    if pd.isna(name):
        return ""

    # to string + lowercase
    name = str(name).lower().strip()

    # replace common punctuation with spaces
    name = re.sub(r"[\,\.\-_/]", " ", name)

    # remove extra spaces first
    name = re.sub(r"\s+", " ", name).strip()

    # list of suffixes to remove as whole words
    # (you can extend this list for your data)
    suffixes = [
        "co ltd", "co ltd", "company limited", "limited",
        "ltd", "inc", "corp", "corporation",
        "llc", "plc", "group", "holding", "holdings",
        "co", "company"
    ]

    # remove suffixes (anywhere, but especially useful at the tail)
    for s in suffixes:
        pattern = r"\b" + re.escape(s) + r"\b"
        name = re.sub(pattern, " ", name)

    # collapse spaces again
    name = re.sub(r"\s+", " ", name).strip()

    return name


# =====================================
# 2. Prepare cleaned columns for both DF
# =====================================
# wos_aff_stdname: reference table
#   - original column: "affiliationame"
# Qichacha_sbus_eng: to be matched
#   - original column: "Qichacha_eng_name"

wos_aff_stdname = wos_aff_stdname.copy()
wos_aff_stdname["clean_affiliation"] = (
    wos_aff_stdname["affiliationame"].astype(str).apply(clean_company_name_en)
)

Qichacha_sbus_eng = Qichacha_sbus_eng.copy()
Qichacha_sbus_eng["clean_qcc_name"] = (
    Qichacha_sbus_eng["Qichacha_eng_name"].astype(str).apply(clean_company_name_en)
)


# ==================================================
# 3. Prepare choices (clean names) for fuzzy matching
# ==================================================
choices_clean = wos_aff_stdname["clean_affiliation"].tolist()


# ===========================================
# 4. Fuzzy matching function (one company)
# ===========================================
def fuzzy_match_one(clean_name_target: str, threshold: int = 80) -> pd.Series:
    """
    Fuzzy-match one cleaned company name against cleaned WOS affiliation list.
    Returns:
        - best matched original affiliationame (or NA)
        - similarity score
    """
    if pd.isna(clean_name_target):
        return pd.Series([pd.NA, 0.0])

    clean_name_target = str(clean_name_target).strip()
    if clean_name_target == "":
        return pd.Series([pd.NA, 0.0])

    match = process.extractOne(
        clean_name_target,
        choices_clean,
        scorer=fuzz.token_sort_ratio
    )

    if match is None:
        return pd.Series([pd.NA, 0.0])

    match_str, score, idx = match

    if score >= threshold:
        matched_original = wos_aff_stdname.iloc[idx]["affiliationame"]
        return pd.Series([matched_original, float(score)])
    else:
        return pd.Series([pd.NA, float(score)])


# =====================================
# 5. Run fuzzy matching on FULL sample
# =====================================
Qichacha_sbus_eng[["affiliationame_match", "match_score"]] = (
    Qichacha_sbus_eng["clean_qcc_name"].progress_apply(fuzzy_match_one)
)

# Optional: see high-quality matches (e.g. score >= 95)
Qichacha_sbus_eng_high = (
    Qichacha_sbus_eng[Qichacha_sbus_eng["match_score"] >= 95]
    .sort_values("match_score", ascending=False)
)

# Inspect
Qichacha_sbus_eng_high

In [ ]:
Qichacha_sbus_eng[Qichacha_sbus_eng["match_score"] >= 90]

In [ ]:
Qichacha_sbus_eng.to_csv(dataset_config['path_processed'] + 'WOS_listed/CN_subs_WOS_match.csv', index=False)

In [ ]:
Qichacha_sbus_eng_high.to_csv(dataset_config['path_processed'] + 'WOS_listed/CN_subs_WOS_highmatch.csv', index=False)